## SRP392258

**paper:** [PMID: 39874782](https://pmc.ncbi.nlm.nih.gov/articles/PMC11810826/) - Goose multi-omics database: A comprehensive multi-omics database for goose genomics, 2025

**date, curator:** 2025-09-03, Sara Carsanaro

**notes**
* each animal has a single SRS ID despite various organs
    * these are not technical replicates -> i didn't mark them as replicates
* organs/tissues were taken from library names in SRA
* age in SRA is 70 days, in methods they refer to the animals as adults. i think annotating as fully formed stage is appropriate as it includes sexually immature and mature
    * 70 days old would be post-fledging and independent, but not sexually mature yet per [animal diversity](https://animaldiversity.org/accounts/Anser_cygnoides/)
* [gooseDB](https://goosedb.com/) is the corresponding website, lists all the organs sampled in this 

### annotation summary
run this after annotation is complete

In [9]:
display_df(organ_df_complete)

,infoOrgan,notes_anat,anatId_,anatName_,anatAnnotationStatus_,anatBiologicalStatus_
0,Abdominal_Fat,,UBERON:0007808,adipose tissue of abdominal region,perfect match,not documented
1,Adrenal_Glands,,UBERON:0002369,adrenal gland,perfect match,not documented
2,Bronchi,,UBERON:0002185,bronchus,perfect match,not documented
3,Bursa_of_Fabricius,,UBERON:0003903,bursa of Fabricius,perfect match,not documented
4,Cecum,,UBERON:0001153,caecum,perfect match,not documented
5,Cloaca,,UBERON:0000162,cloaca,perfect match,not documented
6,Duodenum,,UBERON:0002114,duodenum,perfect match,not documented
7,Esophagus,,UBERON:0001043,esophagus,perfect match,not documented
8,Glandular_Stomach,proventriculus: the glandular or true stomach of a bird that is situated between the crop and gizzard,UBERON:0007357,proventriculus,perfect match,not documented
9,Hair_Follicle,,UBERON:0002073,hair follicle,perfect match,not documented


In [34]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,70 days,UBERON:0000066,fully formed stage,missing child term


### set variables, import packages, define functions

In [4]:
experiment_id = "SRP392258"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)
info_anat_path =  "{}info_anat.tsv".format(path_to_output)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_70998/3311601719.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:11: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████| 102/102 [01:37<00:00,  1.04it/s]
0 samples 

### library annnotations

In [5]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX17072902,SRP392258,HiSeq X Ten,SRS14653380,,,,,,Vas_Deferen,70 days,,,,M,Sichuan White Goose5,,8845,,,,,,M3,SAMN30246267,,,,,,,,,,,03/09/2025,LncRNAseq,,Male biological replicate 2,,,,,,GENOMIC,RANDOM,SRR21057644,E02974B918CF45FD9AB1992C03050252
1,SRX17072901,SRP392258,HiSeq X Ten,SRS14653380,,,,,,Testis,70 days,,,,M,Sichuan White Goose5,,8845,,,,,,M3,SAMN30246267,,,,,,,,,,,03/09/2025,LncRNAseq,,Male biological replicate 2,,,,,,GENOMIC,RANDOM,SRR21057645,32B38A12EB0382DA815863EFA32D9FA4
2,SRX17072900,SRP392258,HiSeq X Ten,SRS14653379,,,,,,Vas_Deferen,70 days,,,,M,Sichuan White Goose4,,8845,,,,,,M2,SAMN30246266,,,,,,,,,,,03/09/2025,LncRNAseq,,Male biological replicate 1,,,,,,GENOMIC,RANDOM,SRR21057646,94769DDB86F4B256F7986176998E5C58
3,SRX17072899,SRP392258,HiSeq X Ten,SRS14653379,,,,,,Testis,70 days,,,,M,Sichuan White Goose4,,8845,,,,,,M2,SAMN30246266,,,,,,,,,,,03/09/2025,LncRNAseq,,Male biological replicate 1,,,,,,GENOMIC,RANDOM,SRR21057647,A46C0599C83C10FDD4D4F3BC6BFADA22
4,SRX17072898,SRP392258,HiSeq X Ten,SRS14653378,,,,,,Ureter,70 days,,,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025,LncRNAseq,,Female biological replicate 3,,,,,,GENOMIC,RANDOM,SRR21057648,A6C9E9A654C7DF7665CFCF5E6745E004
5,SRX17072897,SRP392258,HiSeq X Ten,SRS14653378,,,,,,Trachea,70 days,,,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025,LncRNAseq,,Female biological replicate 3,,,,,,GENOMIC,RANDOM,SRR21057649,D27331337C470AA35349B948EB5D4DB7
6,SRX17072896,SRP392258,HiSeq X Ten,SRS14653378,,,,,,Tongue,70 days,,,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025,LncRNAseq,,Female biological replicate 3,,,,,,GENOMIC,RANDOM,SRR21057650,EFA5D6DD386D43103EE87E0AA249CBBA
7,SRX17072895,SRP392258,HiSeq X Ten,SRS14653378,,,,,,Tail_gland,70 days,,,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025,LncRNAseq,,Female biological replicate 3,,,,,,GENOMIC,RANDOM,SRR21057651,AD8B7FF79F1042EE486A1C24369F8C59
8,SRX17072894,SRP392258,HiSeq X Ten,SRS14653378,,,,,,Subcutaneous_Fat,70 days,,,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025,LncRNAseq,,Female biological replicate 3,,,,,,GENOMIC,RANDOM,SRR21057652,C9E7CF31287AB064ADF7B3CFACF4C1B2
9,SRX17072893,SRP392258,HiSeq X Ten,SRS14653378,,,,,,Spleen,70 days,,,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025,LncRNAseq,,Female biological replicate 3,,,,,,GENOMIC,RANDOM,SRR21057653,34A19A31CF112FB2F3E5CB0A40FB36ED


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [6]:
unique_sorted(library, "infoOrgan")

['Abdominal_Fat' 'Adrenal_Glands' 'Bronchi' 'Bursa_of_Fabricius' 'Cecum'
 'Cloaca' 'Duodenum' 'Esophagus' 'Glandular_Stomach' 'Hair_Follicle'
 'Heart' 'Hypothalamus' 'Jejunum' 'Kidney' 'Leg_Muscle' 'Live' 'Liver'
 'Lung' 'Muscular_Stomach' 'Ovary' 'Oviduct' 'Pancreas' 'Pectoral_muscles'
 'Pineal_gland' 'Pituitary' 'Skin' 'Spleen' 'Subcutaneous_Fat'
 'Tail_gland' 'Testis' 'Tongue' 'Trachea' 'Ureter' 'Vas_Deferen' 'ileum']


In [7]:
infoOrgan = library['infoOrgan'].drop_duplicates().sort_values()
organ_df = pd.DataFrame(infoOrgan)
organ_df = organ_df.reindex(columns=[*organ_df.columns.tolist(), 'notes_anat', 'anatId_', 'anatName_', 'anatAnnotationStatus_', 'anatBiologicalStatus_'], fill_value="")
organ_df.to_csv(info_anat_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# perfect match, missing child term, other
# full sampling, partial sampling, not documented

In [8]:
organ_df_complete = pd.read_csv(info_anat_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN', " "], dtype=object)
display_df(organ_df_complete)

,infoOrgan,notes_anat,anatId_,anatName_,anatAnnotationStatus_,anatBiologicalStatus_
0,Abdominal_Fat,,UBERON:0007808,adipose tissue of abdominal region,perfect match,not documented
1,Adrenal_Glands,,UBERON:0002369,adrenal gland,perfect match,not documented
2,Bronchi,,UBERON:0002185,bronchus,perfect match,not documented
3,Bursa_of_Fabricius,,UBERON:0003903,bursa of Fabricius,perfect match,not documented
4,Cecum,,UBERON:0001153,caecum,perfect match,not documented
5,Cloaca,,UBERON:0000162,cloaca,perfect match,not documented
6,Duodenum,,UBERON:0002114,duodenum,perfect match,not documented
7,Esophagus,,UBERON:0001043,esophagus,perfect match,not documented
8,Glandular_Stomach,proventriculus: the glandular or true stomach of a bird that is situated between the crop and gizzard,UBERON:0007357,proventriculus,perfect match,not documented
9,Hair_Follicle,,UBERON:0002073,hair follicle,perfect match,not documented


In [10]:
library = library.merge(organ_df_complete, left_on='infoOrgan', right_on='infoOrgan')

In [11]:
library['anatId'] = library['anatId_'].values
library['anatName'] = library['anatName_'].values
library['anatAnnotationStatus'] = library['anatAnnotationStatus_'].values
library['anatBiologicalStatus'] = library['anatBiologicalStatus_'].values

In [12]:
library = library[library_cols]
display_df(library)
library.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX17072902,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0001000,vas deferens,,,,Vas_Deferen,70 days,perfect match,not documented,,M,Sichuan White Goose5,,8845,,,,,,M3,SAMN30246267,,,,,,,,,,,03/09/2025
1,SRX17072901,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0000473,testis,,,,Testis,70 days,perfect match,not documented,,M,Sichuan White Goose5,,8845,,,,,,M3,SAMN30246267,,,,,,,,,,,03/09/2025
2,SRX17072900,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0001000,vas deferens,,,,Vas_Deferen,70 days,perfect match,not documented,,M,Sichuan White Goose4,,8845,,,,,,M2,SAMN30246266,,,,,,,,,,,03/09/2025
3,SRX17072899,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0000473,testis,,,,Testis,70 days,perfect match,not documented,,M,Sichuan White Goose4,,8845,,,,,,M2,SAMN30246266,,,,,,,,,,,03/09/2025
4,SRX17072898,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0000056,ureter,,,,Ureter,70 days,perfect match,not documented,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
5,SRX17072897,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0003126,trachea,,,,Trachea,70 days,perfect match,not documented,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
6,SRX17072896,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0001723,tongue,,,,Tongue,70 days,perfect match,not documented,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
7,SRX17072895,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0007802,uropygial gland,,,,Tail_gland,70 days,perfect match,not documented,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
8,SRX17072894,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002190,subcutaneous adipose tissue,,,,Subcutaneous_Fat,70 days,perfect match,not documented,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
9,SRX17072893,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002106,spleen,,,,Spleen,70 days,perfect match,not documented,,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src)

In [13]:
unique_sorted(library, "infoStage")

['70 days']


In [14]:
# all
library.loc[:,'stageId'] = 'UBERON:0000066'
library.loc[:,'stageName'] = 'fully formed stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'missing child term'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX17072902,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan White Goose5,,8845,,,,,,M3,SAMN30246267,,,,,,,,,,,03/09/2025
1,SRX17072901,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan White Goose5,,8845,,,,,,M3,SAMN30246267,,,,,,,,,,,03/09/2025
2,SRX17072900,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan White Goose4,,8845,,,,,,M2,SAMN30246266,,,,,,,,,,,03/09/2025
3,SRX17072899,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan White Goose4,,8845,,,,,,M2,SAMN30246266,,,,,,,,,,,03/09/2025
4,SRX17072898,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0000056,ureter,UBERON:0000066,fully formed stage,,Ureter,70 days,perfect match,not documented,missing child term,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
5,SRX17072897,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0003126,trachea,UBERON:0000066,fully formed stage,,Trachea,70 days,perfect match,not documented,missing child term,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
6,SRX17072896,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0001723,tongue,UBERON:0000066,fully formed stage,,Tongue,70 days,perfect match,not documented,missing child term,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
7,SRX17072895,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0007802,uropygial gland,UBERON:0000066,fully formed stage,,Tail_gland,70 days,perfect match,not documented,missing child term,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
8,SRX17072894,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002190,subcutaneous adipose tissue,UBERON:0000066,fully formed stage,,Subcutaneous_Fat,70 days,perfect match,not documented,missing child term,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
9,SRX17072893,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002106,spleen,UBERON:0000066,fully formed stage,,Spleen,70 days,perfect match,not documented,missing child term,F,Sichuan White Goose3,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [15]:
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

library.loc[:,'strain'] = 'Sichuan white goose'

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX17072902,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,,,,M3,SAMN30246267,,,,,,,,,,,03/09/2025
1,SRX17072901,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,,,,M3,SAMN30246267,,,,,,,,,,,03/09/2025
2,SRX17072900,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,,,,M2,SAMN30246266,,,,,,,,,,,03/09/2025
3,SRX17072899,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,,,,M2,SAMN30246266,,,,,,,,,,,03/09/2025
4,SRX17072898,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0000056,ureter,UBERON:0000066,fully formed stage,,Ureter,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
5,SRX17072897,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0003126,trachea,UBERON:0000066,fully formed stage,,Trachea,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
6,SRX17072896,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0001723,tongue,UBERON:0000066,fully formed stage,,Tongue,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
7,SRX17072895,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0007802,uropygial gland,UBERON:0000066,fully formed stage,,Tail_gland,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
8,SRX17072894,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002190,subcutaneous adipose tissue,UBERON:0000066,fully formed stage,,Subcutaneous_Fat,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
9,SRX17072893,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002106,spleen,UBERON:0000066,fully formed stage,,Spleen,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025


#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [16]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX17072902,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M3,SAMN30246267,,,,,,,,,,,03/09/2025
1,SRX17072901,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M3,SAMN30246267,,,,,,,,,,,03/09/2025
2,SRX17072900,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M2,SAMN30246266,,,,,,,,,,,03/09/2025
3,SRX17072899,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M2,SAMN30246266,,,,,,,,,,,03/09/2025
4,SRX17072898,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0000056,ureter,UBERON:0000066,fully formed stage,,Ureter,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
5,SRX17072897,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0003126,trachea,UBERON:0000066,fully formed stage,,Trachea,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
6,SRX17072896,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0001723,tongue,UBERON:0000066,fully formed stage,,Tongue,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
7,SRX17072895,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0007802,uropygial gland,UBERON:0000066,fully formed stage,,Tail_gland,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
8,SRX17072894,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002190,subcutaneous adipose tissue,UBERON:0000066,fully formed stage,,Subcutaneous_Fat,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025
9,SRX17072893,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002106,spleen,UBERON:0000066,fully formed stage,,Spleen,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,,,,,,,,,,,03/09/2025


#### globin, replicates

In [17]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

      #libraryId        SRSId
101  SRX17072801  SRS14653375
82   SRX17072820  SRS14653375
81   SRX17072821  SRS14653375
21   SRX17072881  SRS14653375
80   SRX17072822  SRS14653375
79   SRX17072823  SRS14653375
78   SRX17072824  SRS14653375
83   SRX17072819  SRS14653375
77   SRX17072825  SRS14653375
75   SRX17072827  SRS14653375
74   SRX17072828  SRS14653375
32   SRX17072870  SRS14653375
65   SRX17072837  SRS14653375
43   SRX17072859  SRS14653375
100  SRX17072802  SRS14653375
54   SRX17072848  SRS14653375
84   SRX17072818  SRS14653375
76   SRX17072826  SRS14653375
86   SRX17072816  SRS14653375
85   SRX17072817  SRS14653375
96   SRX17072806  SRS14653375
95   SRX17072807  SRS14653375
94   SRX17072808  SRS14653375
93   SRX17072809  SRS14653375
92   SRX17072810  SRS14653375
99   SRX17072803  SRS14653375
10   SRX17072892  SRS14653375
90   SRX17072812  SRS14653375
89   SRX17072813  SRS14653375
88   SRX17072814  SRS14653375
87   SRX17072815  SRS14653375
91   SRX17072811  SRS14653375
98   SRX17

/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_70998/3311601719.py:43: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dups = df[duplicateCheck].loc[:,['#libraryId', column]]


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# i don't think these are actually technical replicates

# replicates
#library.loc[library["#SRSId"] == "SRS14653375", "replicate"] = "1"
#library.loc[library["#SRSId"] == "SRS14653376", "replicate"] = "2"
#library.loc[library["#SRSId"] == "SRS14653377", "replicate"] = "3"
#library.loc[library["#SRSId"] == "SRS14653378", "replicate"] = "4"
#library.loc[library["#SRSId"] == "SRS14653379", "replicate"] = "5"
#library.loc[library["#SRSId"] == "SRS14653380", "replicate"] = "6"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status

In [18]:
library.loc[:,'sampleAge_value'] = '70'
library.loc[:,'sampleAge_unit'] = 'days'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX17072902,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M3,SAMN30246267,70,days,,,,,,,,,03/09/2025
1,SRX17072901,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M3,SAMN30246267,70,days,,,,,,,,,03/09/2025
2,SRX17072900,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M2,SAMN30246266,70,days,,,,,,,,,03/09/2025
3,SRX17072899,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M2,SAMN30246266,70,days,,,,,,,,,03/09/2025
4,SRX17072898,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0000056,ureter,UBERON:0000066,fully formed stage,,Ureter,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,,03/09/2025
5,SRX17072897,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0003126,trachea,UBERON:0000066,fully formed stage,,Trachea,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,,03/09/2025
6,SRX17072896,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0001723,tongue,UBERON:0000066,fully formed stage,,Tongue,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,,03/09/2025
7,SRX17072895,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0007802,uropygial gland,UBERON:0000066,fully formed stage,,Tail_gland,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,,03/09/2025
8,SRX17072894,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002190,subcutaneous adipose tissue,UBERON:0000066,fully formed stage,,Subcutaneous_Fat,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,,03/09/2025
9,SRX17072893,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002106,spleen,UBERON:0000066,fully formed stage,,Spleen,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,,03/09/2025


#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [28]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2025-09-04'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX17072902,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M3,SAMN30246267,70,days,,,,,,,,SAC,2025-09-04
1,SRX17072901,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M3,SAMN30246267,70,days,,,,,,,,SAC,2025-09-04
2,SRX17072900,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M2,SAMN30246266,70,days,,,,,,,,SAC,2025-09-04
3,SRX17072899,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M2,SAMN30246266,70,days,,,,,,,,SAC,2025-09-04
4,SRX17072898,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0000056,ureter,UBERON:0000066,fully formed stage,,Ureter,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,SAC,2025-09-04
5,SRX17072897,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0003126,trachea,UBERON:0000066,fully formed stage,,Trachea,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,SAC,2025-09-04
6,SRX17072896,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0001723,tongue,UBERON:0000066,fully formed stage,,Tongue,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,SAC,2025-09-04
7,SRX17072895,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0007802,uropygial gland,UBERON:0000066,fully formed stage,,Tail_gland,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,SAC,2025-09-04
8,SRX17072894,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002190,subcutaneous adipose tissue,UBERON:0000066,fully formed stage,,Subcutaneous_Fat,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,SAC,2025-09-04
9,SRX17072893,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002106,spleen,UBERON:0000066,fully formed stage,,Spleen,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,,,,SAC,2025-09-04


#### comments

In [29]:
library.loc[:,'comment'] = 'SRA has age as 70 days but paper refers to them as adults - kept as 70 days in sampleAge and infoStage but used fully formed stage for annotation to capture both'

#### save complete file with correct columns

In [30]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX17072902,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M3,SAMN30246267,70,days,,,,,SRA has age as 70 days but paper refers to them as adults - kept as 70 days in sampleAge and infoStage but used fully formed stage for annotation to capture both,,,SAC,2025-09-04
1,SRX17072901,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M3,SAMN30246267,70,days,,,,,SRA has age as 70 days but paper refers to them as adults - kept as 70 days in sampleAge and infoStage but used fully formed stage for annotation to capture both,,,SAC,2025-09-04
2,SRX17072900,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M2,SAMN30246266,70,days,,,,,SRA has age as 70 days but paper refers to them as adults - kept as 70 days in sampleAge and infoStage but used fully formed stage for annotation to capture both,,,SAC,2025-09-04
3,SRX17072899,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M2,SAMN30246266,70,days,,,,,SRA has age as 70 days but paper refers to them as adults - kept as 70 days in sampleAge and infoStage but used fully formed stage for annotation to capture both,,,SAC,2025-09-04
4,SRX17072898,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0000056,ureter,UBERON:0000066,fully formed stage,,Ureter,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,SRA has age as 70 days but paper refers to them as adults - kept as 70 days in sampleAge and infoStage but used fully formed stage for annotation to capture both,,,SAC,2025-09-04
5,SRX17072897,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0003126,trachea,UBERON:0000066,fully formed stage,,Trachea,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,SRA has age as 70 days but paper refers to them as adults - kept as 70 days in sampleAge and infoStage but used fully formed stage for annotation to capture both,,,SAC,2025-09-04
6,SRX17072896,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0001723,tongue,UBERON:0000066,fully formed stage,,Tongue,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,SRA has age as 70 days but paper refers to them as adults - kept as 70 days in sampleAge and infoStage but used fully formed stage for annotation to capture both,,,SAC,2025-09-04
7,SRX17072895,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0007802,uropygial gland,UBERON:0000066,fully formed stage,,Tail_gland,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,SRA has age as 70 days but paper refers to them as adults - kept as 70 days in sampleAge and infoStage but used fully formed stage for annotation to capture both,,,SAC,2025-09-04
8,SRX17072894,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0002190,subcutaneous adipose tissue,UBERON:0000066,fully formed stage,,Subcutaneous_Fat,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246

### experiment annotations

In [21]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP392258,Transcriptome atlas of 34 goose tissues,"The studies of goose gene expression and evolution among different vertebrates by comparative transcriptome, not only help us in deeply understanding theevolutionary relationships, but also promote us in revealing the molecular geneticmechanism on various characteristics or traits of different species. We collected 34 tissues to perform lncRNAseq to explore the characteristics of goose.",SRA,,,,,,,PRJNA868424,39874782,,10.1016/j.psj.2025.104842,,


#### experiment and protocol details

In [22]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

102

In [23]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
#experiment.loc[:,'projectTags'] = '' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP392258,Transcriptome atlas of 34 goose tissues,"The studies of goose gene expression and evolution among different vertebrates by comparative transcriptome, not only help us in deeply understanding theevolutionary relationships, but also promote us in revealing the molecular geneticmechanism on various characteristics or traits of different species. We collected 34 tissues to perform lncRNAseq to explore the characteristics of goose.",SRA,total,,102,,,,PRJNA868424,39874782,,10.1016/j.psj.2025.104842,,


#### paper and xrefs

In [24]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11810826/'
#experiment.loc[:,'DOI'] = ''
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP392258,Transcriptome atlas of 34 goose tissues,"The studies of goose gene expression and evolution among different vertebrates by comparative transcriptome, not only help us in deeply understanding theevolutionary relationships, but also promote us in revealing the molecular geneticmechanism on various characteristics or traits of different species. We collected 34 tissues to perform lncRNAseq to explore the characteristics of goose.",SRA,total,,102,,,,PRJNA868424,39874782,https://pmc.ncbi.nlm.nih.gov/articles/PMC11810826/,10.1016/j.psj.2025.104842,,


#### comments

In [26]:
experiment.loc[:,'comment'] = 'did not mark libraries with same SRSid as replicates because they are from different tissues, they are not technical replicates'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP392258,Transcriptome atlas of 34 goose tissues,"The studies of goose gene expression and evolution among different vertebrates by comparative transcriptome, not only help us in deeply understanding theevolutionary relationships, but also promote us in revealing the molecular geneticmechanism on various characteristics or traits of different species. We collected 34 tissues to perform lncRNAseq to explore the characteristics of goose.",SRA,total,,102,,,,PRJNA868424,39874782,https://pmc.ncbi.nlm.nih.gov/articles/PMC11810826/,10.1016/j.psj.2025.104842,,"did not mark libraries with same SRSid as replicates because they are from different tissues, they are not technical replicates"


#### save complete file

In [27]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [32]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [33]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [35]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
49541,SRX18248090,SRP407711,Illumina NovaSeq 6000,SRS15743606,UBERON:0002535,gill,UBERON:0000066,fully formed stage,,gill,,perfect match,partial sampling,other,NA,,,98310,,,ribo-minus,,9,C3,SAMN31710378,,,,,,,"PMID:36671800, For mRNA, lncRNA, and circRNA s...",salinity 40 ppt,,ANN,2025-09-02
49542,SRX18248083,SRP407711,Illumina NovaSeq 6000,SRS15743606,UBERON:0002535,gill,UBERON:0000066,fully formed stage,,gill,,perfect match,partial sampling,other,NA,,,98310,VAHTSTM Small RNA Library Prep Kit,,miRNA,,9,C3,SAMN31710378,,,,,,,"PMID:36671800, For mRNA, lncRNA, and circRNA s...",salinity 40 ppt,,ANN,2025-09-02
49543,SRX17072902,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M3,SAMN30246267,70,days,,,,,SRA has age as 70 days but paper refers to the...,,,SAC,2025-09-04
49544,SRX17072901,SRP392258,HiSeq X Ten,SRS14653380,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M3,SAMN30246267,70,days,,,,,SRA has age as 70 days but paper refers to the...,,,SAC,2025-09-04
49545,SRX17072900,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0001000,vas deferens,UBERON:0000066,fully formed stage,,Vas_Deferen,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M2,SAMN30246266,70,days,,,,,SRA has age as 70 days but paper refers to the...,,,SAC,2025-09-04
49546,SRX17072899,SRP392258,HiSeq X Ten,SRS14653379,UBERON:0000473,testis,UBERON:0000066,fully formed stage,,Testis,70 days,perfect match,not documented,missing child term,M,Sichuan white goose,,8845,,,polyA,,,M2,SAMN30246266,70,days,,,,,SRA has age as 70 days but paper refers to the...,,,SAC,2025-09-04
49547,SRX17072898,SRP392258,HiSeq X Ten,SRS14653378,UBERON:0000056,ureter,UBERON:0000066,fully formed stage,,Ureter,70 days,perfect match,not documented,missing child term,F,Sichuan white goose,,8845,,,polyA,,,F4,SAMN30246265,70,days,,,,,SRA has age as 70 days but paper refers to the...,,,SAC,2025-09-04


In [36]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
921,SRP580352,Razor clam salinity transcriptome,Transcriptome data of different tissues in raz...,SRA,total,Bgee 1K,27,,full_length,,PRJNA1253863,40865426,https://www.sciencedirect.com/science/article/...,10.1016/j.marenvres.2025.107467,,
922,SRP407711,Sinonovacula constricta whole-transcriptome,The results will provide insights into the reg...,SRA,total,Bgee 1K,18,VAHTSTM Small RNA Library Prep Kit,,,PRJNA901274,36671800,https://pmc.ncbi.nlm.nih.gov/articles/PMC9856061/,10.3390/biology12010106,,a protocol is cited for the ‘nine small RNA li...
923,SRP392258,Transcriptome atlas of 34 goose tissues,The studies of goose gene expression and evolu...,SRA,total,,102,,,,PRJNA868424,39874782,https://pmc.ncbi.nlm.nih.gov/articles/PMC11810...,10.1016/j.psj.2025.104842,,did not mark libraries with same SRSid as repl...


### add annotations to git

In [37]:
! git pull

Already up to date.


In [38]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [39]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [40]:
! git add $git_experiment_path $git_library_path

In [41]:
! git commit -m $commit_message_exp

[develop 50dee42] adding annotated bulk experiment SRP392258
 2 files changed, 103 insertions(+)


In [42]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.72 KiB | 3.72 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git
   2e6cfdf..50dee42  develop -> develop


### add annotation folder and script to git

In [ ]:
! git status

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push